# Diagnóstico de acesso aos dados FACEIT
## Resultado
A autenticação funcionou. A sonda recuperou três partidas de julho de 2026 e 30 registros jogador-partida.
Os campos consultados não incluem FACEIT Rating, Swing ou Impact.
O ranking final Challenger EU da Season 8 ainda não foi recuperado.

Este notebook verifica as evidências locais. Não consulta a rede, não lê a chave e não calcula outliers.


## Contexto e método
Objetivo final: comparar a média do FACEIT Rating dos jogadores que terminaram a Season 8 como Challenger EU.

A sonda selecionou três partidas de donk666, sem amostragem aleatória, em uma janela interna à season.
Essas partidas servem apenas para investigar o formato e a disponibilidade dos dados.

### Hipóteses e limites
- Janela de consulta: 1º de maio a 31 de julho de 2026, UTC. Não representa os limites exatos da season.
- Histórico usa segundos Unix; estatísticas por jogador usam milissegundos.
- Ausência de um campo nestas respostas não prova ausência em todos os serviços da FACEIT.
- Nenhuma lista atual de jogadores foi tratada como classificação final da Season 8.
- A igualdade dos nomes de seasons precisa ser validada pelas datas.


In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data/probes").exists():
    ROOT = ROOT.parent
PROBE = ROOT / 'data/probes/20261001T024717084718Z'

def read_json(name):
    return json.loads((PROBE / name).read_text(encoding="utf-8"))

summary = read_json("summary.json")
print("Coleta UTC:", summary["collected_at_utc"])
print("Evidências:", PROBE)


Coleta UTC: 2026-10-01T02:47:17.084718+00:00
Evidências: C:\Users\report\Documents\GitHub\CS2-Outlier-Detector\data\probes\20261001T024717084718Z


## Dados
### Respostas da API
Todas as requisições abaixo são consultas de leitura; os arquivos não contêm cabeçalhos de autenticação.

In [2]:
requests = summary["requests"] + read_json("additional_requests.json")
request_table = pd.DataFrame(requests)
assert request_table["status"].eq(200).all()
request_table[["label", "status", "file"]]


,label,status,file
0,current_ranking,200,current_ranking.json
1,api_spec,200,api_spec.json
2,player,200,player.json
3,player_stats_window,200,player_stats_window.json
4,history_window,200,history_window.json
5,match_stats_1,200,match_stats_1.json
6,match_stats_2,200,match_stats_2.json
7,match_stats_3,200,match_stats_3.json
8,matchmaking_details,200,matchmaking_details.json
9,matchmaking_league,200,matchmaking_league.json


### Partidas e datas
Conferência da janela temporal e dos identificadores retornados pelas duas consultas.

In [3]:
history = pd.DataFrame(read_json("history_window.json")["items"])
history["finished_utc"] = pd.to_datetime(history["finished_at"], unit="s", utc=True)
start = pd.Timestamp(summary["probe_window"]["start"])
end = pd.Timestamp(summary["probe_window"]["end"])
assert history["finished_utc"].between(start, end).all()
assert history["match_id"].is_unique
player_stats = pd.DataFrame([item["stats"] for item in read_json("player_stats_window.json")["items"]])
player_stats["finished_utc"] = pd.to_datetime(player_stats["Match Finished At"].astype("int64"), unit="ms", utc=True)
assert player_stats["finished_utc"].between(start, end).all()
assert set(player_stats["Match Id"]) == set(history["match_id"])
joined = player_stats.merge(history[["match_id", "finished_utc"]], left_on="Match Id", right_on="match_id", validate="one_to_one", suffixes=("_stats", "_history"))
assert joined["finished_utc_stats"].eq(joined["finished_utc_history"]).all()
history[["match_id", "finished_utc", "competition_name", "competition_type"]]


,match_id,finished_utc,competition_name,competition_type
0,1-02f245cf-0f54-40d0-8b5e-0aee19d8662c,2026-07-31 18:33:56+00:00,Europe 5v5 Queue,matchmaking
1,1-dfe6292c-5fc0-4f2d-89c2-5426ed5eb8aa,2026-07-30 21:50:26+00:00,Europe 5v5 Queue,matchmaking
2,1-a7d5a458-bd57-4902-b6e3-ac0347d918a6,2026-07-30 20:04:00+00:00,Europe 5v5 Queue,matchmaking


## Resultados
### Campos disponíveis por jogador-partida
Cada registro representa um jogador em um mapa da partida. Repetições do mesmo jogador em partidas diferentes são esperadas.

In [4]:
records = []
for request in summary["requests"]:
    if not request["label"].startswith("match_stats_") or request.get("status") != 200:
        continue
    payload = read_json(request["file"])
    for round_index, round_data in enumerate(payload.get("rounds", [])):
        for team in round_data.get("teams", []):
            for player in team.get("players", []):
                records.append({
                    "match_id": round_data.get("match_id"),
                    "map_index": round_index,
                    "player_id": player.get("player_id"),
                    "nickname": player.get("nickname"),
                    **player.get("player_stats", {}),
                })
matches = pd.DataFrame(records)
keys = ["match_id", "map_index", "player_id"]
assert matches[keys].notna().all().all()
assert not matches.duplicated(keys).any()
assert set(matches["match_id"]) == set(history["match_id"])
rating_candidates = [field for field in matches.columns if any(term in field.lower() for term in ("rating", "swing", "impact"))]
print("Partidas:", matches["match_id"].nunique())
print("Registros jogador-partida:", len(matches))
print("Campos candidatos a rating:", rating_candidates)
print("Campos nas estatísticas por jogador:", len(player_stats.columns) - 1)
matches.groupby("match_id").agg(registros=("player_id", "size"), jogadores=("player_id", "nunique"))


Partidas: 3
Registros jogador-partida: 30
Campos candidatos a rating: []
Campos nas estatísticas por jogador: 36


,registros,jogadores
match_id,,
1-02f245cf-0f54-40d0-8b5e-0aee19d8662c,10,10
1-a7d5a458-bd57-4902-b6e3-ac0347d918a6,10,10
1-dfe6292c-5fc0-4f2d-89c2-5426ed5eb8aa,10,10


### Estatísticas de desempenho retornadas
Essas métricas são observadas na API. Não substituem o FACEIT Rating pretendido no projeto.

In [5]:
metrics = ["ADR", "K/D Ratio", "Kills", "Deaths", "Assists"]
for field in metrics:
    numeric = pd.to_numeric(matches[field], errors="coerce")
    assert numeric.notna().all()
    assert numeric.ge(0).all()
matches.loc[matches["player_id"].eq(summary["player"]["player_id"]), ["nickname", "match_id"] + metrics]


,nickname,match_id,ADR,K/D Ratio,Kills,Deaths,Assists
5,donk666,1-02f245cf-0f54-40d0-8b5e-0aee19d8662c,128,2.86,20,7,3
11,donk666,1-dfe6292c-5fc0-4f2d-89c2-5426ed5eb8aa,102.6,1.71,24,14,4
26,donk666,1-a7d5a458-bd57-4902-b6e3-ac0347d918a6,109.4,1.4,21,15,6


### Ranking e significado de season
A especificação do ranking não oferece parâmetro temporal. A season 8 da liga vinculada à fila EU corresponde a maio de 2024 e não pode ser usada como a Season 8 de CS2.

In [6]:
spec = read_json("api_spec.json")
path = "/rankings/games/{game_id}/regions/{region}"
parameters = [item["name"] for item in spec["paths"][path]["get"]["parameters"]]
print("Parâmetros documentados do ranking:", parameters)
league_season = read_json("league_season_8_candidate.json")["season"]
print("Season da liga consultada:", league_season["number"])
print("Início:", league_season["start_date"])
print("Fim:", league_season["end_date"])
assert pd.Timestamp(league_season["end_date"]) < start


Parâmetros documentados do ranking: ['game_id', 'region', 'country', 'offset', 'limit']
Season da liga consultada: 8
Início: 2024-05-01T00:00:00Z
Fim: 2024-06-01T00:00:00Z


## Conclusões e próximos passos
- **Autenticação e histórico:** funcionam nas consultas executadas.
- **Rating:** não foi exposto nos campos dos 30 registros jogador-partida inspecionados, nem nas três estatísticas individuais. Isso impede calcular a média pretendida com esta amostra.
- **Coorte histórica:** ainda falta a classificação final dos Challenger EU da Season 8.
- **Cobertura:** não foi testada uma extração de todos os jogadores ou todas as partidas.
- **Próxima ação:** solicitar à FACEIT o endpoint suportado ou uma exportação de ambas as fontes, além dos limites UTC e da regra de agregação da média.
- ADR e K/D existem nas respostas, mas uma troca de métrica exige uma decisão do projeto.

## Fontes
- [Data API](https://docs.faceit.com/docs/data-api/data/)
- [FACEIT Rating](https://support.faceit.com/hc/en-us/articles/26530513602716-FACEIT-Season-8-FACEIT-Rating-FAQ)
- [Início da Season 8](https://support.faceit.com/hc/en-us/articles/26259323321244-FACEIT-Season-8-Placement-Matches-Soft-Reset-FAQ)
- [Perfis por season](https://support.faceit.com/hc/en-us/articles/28929290086556-Seasonal-Stats-FAQ)

As respostas e URLs exatas das consultas estão na pasta de evidências indicada no início.
